**Create BigQuery Table**

In [1]:
from google.cloud import bigquery
from google.oauth2 import service_account

# =========================================
# AUTHENTICATION
# =========================================

SERVICE_ACCOUNT_FILE = "../Keys/BigQueryKey.json"

credentials = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE
)

PROJECT_ID = "depihealthnux"
DATASET_ID = "Depihealthnux_Main"
TABLE_ID = "Payments"

TABLE_REF = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"

client = bigquery.Client(
    credentials=credentials,
    project=PROJECT_ID
)

# =========================================
# SCHEMA
# =========================================

schema = [

    bigquery.SchemaField(
        "Payment_Key",
        "STRING",
        mode="REQUIRED"
    ),

    bigquery.SchemaField(
        "Booking_Key",
        "STRING"
    ),

    bigquery.SchemaField(
        "Patient_U_ID",
        "STRING"
    ),

    bigquery.SchemaField(
        "Payment_Date",
        "DATE"
    ),

    bigquery.SchemaField(
        "Payment_Type",
        "STRING"
    ),

    bigquery.SchemaField(
        "Payment_Amount",
        "INTEGER"
    ),

    bigquery.SchemaField(
        "Payment_Status",
        "STRING"
    )

]

table = bigquery.Table(
    TABLE_REF,
    schema=schema
)

table.clustering_fields = [
    "Patient_U_ID",
    "Booking_Key"
]

client.create_table(
    table,
    exists_ok=True
)

print("=================================")
print("BigQuery table created successfully")
print(TABLE_REF)
print("=================================")

BigQuery table created successfully
depihealthnux.Depihealthnux_Main.Payments


**Create Postgres table**

In [2]:
import sys
import psycopg2

sys.path.append("..")
from Keys.PostGresKey import POSTGRES_URL

# =========================================
# CONNECT
# =========================================

conn = psycopg2.connect(POSTGRES_URL)
cursor = conn.cursor()

# =========================================
# CREATE SEQUENCE
# =========================================

cursor.execute("""

CREATE SEQUENCE IF NOT EXISTS payment_no_seq;

""")

# =========================================
# CREATE TABLE
# =========================================

create_table_query = """
CREATE TABLE IF NOT EXISTS Payments (

    Payment_Key TEXT PRIMARY KEY
    DEFAULT (
        'Pay' ||
        LPAD(
            nextval('payment_no_seq')::TEXT,
            4,
            '0'
        )
    ),

    Booking_Key TEXT,

    Patient_U_ID TEXT,

    Payment_Date DATE,

    Payment_Type TEXT,

    Payment_Amount INTEGER,

    Payment_Status TEXT

);
"""

cursor.execute(create_table_query)

# =========================================
# INDEXES
# =========================================

cursor.execute("""

CREATE INDEX IF NOT EXISTS idx_payment_booking
ON Payments(Booking_Key);

""")

cursor.execute("""

CREATE INDEX IF NOT EXISTS idx_payment_patient
ON Payments(Patient_U_ID);

""")

cursor.execute("""

CREATE INDEX IF NOT EXISTS idx_payment_status
ON Payments(Payment_Status);

""")

conn.commit()

print("=================================")
print("PostgreSQL table created successfully")
print("Table: Payments")
print("=================================")

cursor.close()
conn.close()

PostgreSQL table created successfully
Table: Payments


**Sync BigQuery to Postgres**

In [3]:
import sys
import pandas as pd
import psycopg2

from psycopg2.extras import execute_values

from google.cloud import bigquery
from google.oauth2 import service_account

sys.path.append("..")
from Keys.PostGresKey import POSTGRES_URL

# =========================================
# BIGQUERY AUTH
# =========================================

credentials = service_account.Credentials.from_service_account_file(
    "../Keys/BigQueryKey.json"
)

client = bigquery.Client(
    credentials=credentials,
    project="depihealthnux"
)

# =========================================
# READ BIGQUERY
# =========================================

query = """

SELECT

    Payment_Key,
    Booking_Key,
    Patient_U_ID,
    Payment_Date,
    Payment_Type,
    Payment_Amount,
    Payment_Status

FROM `depihealthnux.Depihealthnux_Main.Payments`

ORDER BY Payment_Key

"""

df = client.query(query).to_dataframe()

print(df.head(3))
print(f"Rows Retrieved: {len(df)}")

# =========================================
# CONNECT POSTGRES
# =========================================

conn = psycopg2.connect(POSTGRES_URL)
cursor = conn.cursor()

cursor.execute("""
DELETE FROM Payments;
""")

if not df.empty:

    execute_values(
        cursor,
        """
        INSERT INTO Payments (

            Payment_Key,
            Booking_Key,
            Patient_U_ID,
            Payment_Date,
            Payment_Type,
            Payment_Amount,
            Payment_Status

        )
        VALUES %s
        """,
        df.values.tolist(),
        page_size=1000
    )

# =========================================
# RESET SEQUENCE
# =========================================

cursor.execute("""

SELECT setval(
    'payment_no_seq',
    COALESCE(
        (
            SELECT MAX(
                CAST(
                    REPLACE(Payment_Key,'Pay','')
                    AS INTEGER
                )
            )
            FROM Payments
        ),
        1
    ),
    true
);

""")

conn.commit()

print(f"Inserted {len(df)} rows")

cursor.execute("""

SELECT *
FROM Payments
ORDER BY Payment_Key
LIMIT 3

""")

print("\nFirst 3 Rows From PostgreSQL")

for row in cursor.fetchall():
    print(row)

cursor.close()
conn.close()

Empty DataFrame
Columns: [Payment_Key, Booking_Key, Patient_U_ID, Payment_Date, Payment_Type, Payment_Amount, Payment_Status]
Index: []
Rows Retrieved: 0
Inserted 0 rows

First 3 Rows From PostgreSQL


**Sync Postgres to BigQuery**

In [4]:
import sys
import pandas as pd
import psycopg2

from google.cloud import bigquery
from google.oauth2 import service_account

sys.path.append("..")
from Keys.PostGresKey import POSTGRES_URL

# =========================================
# BIGQUERY AUTH
# =========================================

credentials = service_account.Credentials.from_service_account_file(
    "../Keys/BigQueryKey.json"
)

client = bigquery.Client(
    credentials=credentials,
    project="depihealthnux"
)

# =========================================
# READ POSTGRES
# =========================================

conn = psycopg2.connect(POSTGRES_URL)

query = """

SELECT

    Payment_Key,
    Booking_Key,
    Patient_U_ID,
    Payment_Date,
    Payment_Type,
    Payment_Amount,
    Payment_Status

FROM Payments

ORDER BY Payment_Key

"""

df = pd.read_sql(query, conn)

print(df.head(3))
print(f"Rows Retrieved: {len(df)}")

conn.close()

# =========================================
# LOAD TO BIGQUERY
# =========================================

job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE"
)

job = client.load_table_from_dataframe(
    df,
    "depihealthnux.Depihealthnux_Main.Payments",
    job_config=job_config
)

job.result()

print(f"Uploaded {len(df)} rows")

verify_df = client.query("""

SELECT *
FROM `depihealthnux.Depihealthnux_Main.Payments`
LIMIT 3

""").to_dataframe()

print("\nFirst 3 Rows From BigQuery")

print(verify_df)

C:\Users\Sedeek Elmasry\AppData\Local\Temp\ipykernel_6500\2268899764.py:48: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Empty DataFrame
Columns: [payment_key, booking_key, patient_u_id, payment_date, payment_type, payment_amount, payment_status]
Index: []
Rows Retrieved: 0
Uploaded 0 rows

First 3 Rows From BigQuery
Empty DataFrame
Columns: [payment_key, booking_key, patient_u_id, payment_date, payment_type, payment_amount, payment_status]
Index: []
